In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif


In [ ]:
Resume = pd.read_csv("Resume.csv")
Jobs = pd.read_csv("Jobs.csv")
matches = pd.read_csv("matches.csv")

print("Resume:", Resume.shape)
print("Jobs:", Jobs.shape)
print("Matches:", matches.shape)


In [ ]:
print("Resume Dataset")
print(Resume.info())

print("\nJobs Dataset")
print(Jobs.info())

print("\nMatches Dataset")
print(matches.info())


In [ ]:
print("Resume Columns:")
print(Resume.columns.tolist())

print("\nJobs Columns:")
print(Jobs.columns.tolist())

print("\nMatches Columns:")
print(matches.columns.tolist())


In [ ]:
print("Resume Summary Statistics")
display(Resume.describe(include="all").T)

print("\nJobs Summary Statistics")
display(Jobs.describe(include="all").T)


In [ ]:
print("Missing Values in Resume:")
print(Resume.isnull().sum())

print("\nMissing Values in Jobs:")
print(Jobs.isnull().sum())

print("\nMissing Values in Matches:")
print(matches.isnull().sum())


In [ ]:
print("Duplicate rows in Resume:", Resume.duplicated().sum())
print("Duplicate rows in Jobs:", Jobs.duplicated().sum())
print("Duplicate rows in Matches:", matches.duplicated().sum())


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(Resume["years_experience"], bins=10, kde=True)
plt.title("Distribution of Years of Experience")
plt.xlabel("Years of Experience")
plt.ylabel("Number of Resumes")
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=Resume, x="seniority")
plt.title("Resume Seniority Distribution")
plt.xlabel("Seniority")
plt.ylabel("Number of Resumes")
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
Resume["role"].value_counts().head(10).plot(kind="bar")
plt.title("Top 10 Resume Roles")
plt.xlabel("Role")
plt.ylabel("Number of Resumes")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
Resume["industry"].value_counts().plot(kind="bar")
plt.title("Resume Distribution by Industry")
plt.xlabel("Industry")
plt.ylabel("Number of Resumes")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
Resume["education"].value_counts().plot(kind="bar")
plt.title("Education Distribution")
plt.xlabel("Education")
plt.ylabel("Number of Resumes")
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=Jobs, x="seniority")
plt.title("Job Seniority Distribution")
plt.xlabel("Seniority")
plt.ylabel("Number of Jobs")
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
Jobs["industry"].value_counts().plot(kind="bar")
plt.title("Jobs by Industry")
plt.xlabel("Industry")
plt.ylabel("Number of Jobs")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
Jobs["job_title"].value_counts().head(10).plot(kind="bar")
plt.title("Top 10 Job Titles")
plt.xlabel("Job Title")
plt.ylabel("Number of Jobs")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=Resume["years_experience"])
plt.title("Boxplot of Years of Experience")
plt.xlabel("Years of Experience")
plt.show()


In [ ]:
numeric_data = Resume.select_dtypes(include=np.number)

display(numeric_data.corr())

plt.figure(figsize=(6, 4))
sns.heatmap(numeric_data.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
Resume = Resume.copy()
Jobs = Jobs.copy()
matches = matches.copy()

for col in Resume.select_dtypes(include="object").columns:
    Resume[col] = Resume[col].fillna(Resume[col].mode()[0])

for col in Jobs.select_dtypes(include="object").columns:
    Jobs[col] = Jobs[col].fillna(Jobs[col].mode()[0])

for col in matches.select_dtypes(include="object").columns:
    matches[col] = matches[col].fillna(matches[col].mode()[0])

Resume["years_experience"] = Resume["years_experience"].fillna(
    Resume["years_experience"].median()
)

print("Missing values handled.")


In [ ]:
Resume = Resume.drop_duplicates().reset_index(drop=True)
Jobs = Jobs.drop_duplicates().reset_index(drop=True)
matches = matches.drop_duplicates().reset_index(drop=True)

print("Duplicates removed.")


In [ ]:
Q1 = Resume["years_experience"].quantile(0.25)
Q3 = Resume["years_experience"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

Resume["years_experience"] = Resume["years_experience"].clip(lower, upper)

print("Outliers treated using IQR capping.")


In [ ]:
skewness = Resume["years_experience"].skew()

print("Skewness before treatment:", skewness)

if abs(skewness) > 0.5:
    Resume["years_experience"] = np.log1p(Resume["years_experience"])

print("Skewness treatment completed.")


In [ ]:
def get_skills(value):
    value = str(value)
    value = value.replace("[", "").replace("]", "").replace("'", "")
    return [x.strip().lower() for x in value.split(",") if x.strip()]

Resume["skills_list"] = Resume["skills"].apply(get_skills)
Jobs["required_skills_list"] = Jobs["must_have_skills"].apply(get_skills)

Resume["skill_count"] = Resume["skills_list"].apply(len)
Jobs["required_skill_count"] = Jobs["required_skills_list"].apply(len)

print(Resume[["skills_list", "skill_count"]].head())
print(Jobs[["required_skills_list", "required_skill_count"]].head())


In [ ]:
resume_dict = Resume.set_index("resume_id").to_dict("index")
job_dict = Jobs.set_index("job_id").to_dict("index")

pairs = []

for _, row in matches.iterrows():
    job_id = row["job_id"]
    relevant_ids = get_skills(row["relevant_resume_ids"])

    for resume_id in relevant_ids:
        if resume_id in resume_dict:
            pairs.append([resume_id, job_id, 1])

    non_relevant = [
        r for r in Resume["resume_id"]
        if r not in relevant_ids
    ]

    sample_size = min(len(relevant_ids), len(non_relevant))

    for resume_id in np.random.choice(non_relevant, sample_size, replace=False):
        pairs.append([resume_id, job_id, 0])

pairs = pd.DataFrame(
    pairs,
    columns=["resume_id", "job_id", "target"]
)

print("Resume-Job pairs:", pairs.shape)
print(pairs["target"].value_counts())


In [ ]:
data = pairs.merge(
    Resume,
    on="resume_id",
    how="left"
)

data = data.merge(
    Jobs,
    on="job_id",
    how="left",
    suffixes=("_resume", "_job")
)

print("Merged dataset shape:", data.shape)


In [ ]:
def skill_match(row):
    resume_skills = set(row["skills_list"])
    job_skills = set(row["required_skills_list"])
    return len(resume_skills.intersection(job_skills))

data["skill_match_count"] = data.apply(skill_match, axis=1)

data["skill_match_ratio"] = (
    data["skill_match_count"] / data["required_skill_count"].replace(0, 1)
)

data["seniority_match"] = (
    data["seniority_resume"] == data["seniority_job"]
).astype(int)

data["industry_match"] = (
    data["industry_resume"] == data["industry_job"]
).astype(int)

display(
    data[
        [
            "skill_match_count",
            "skill_match_ratio",
            "seniority_match",
            "industry_match",
            "target"
        ]
    ].head()
)


In [ ]:
le = LabelEncoder()

data["seniority_resume_encoded"] = le.fit_transform(
    data["seniority_resume"]
)

data["seniority_job_encoded"] = le.transform(
    data["seniority_job"]
)

print("Label encoding completed.")


In [ ]:
categorical_columns = [
    "role",
    "industry_resume",
    "education",
    "job_title",
    "industry_job"
]

encoded_data = pd.get_dummies(
    data[categorical_columns],
    drop_first=True
)

print("One-hot encoded shape:", encoded_data.shape)


In [ ]:
numeric_columns = [
    "years_experience",
    "skill_count",
    "required_skill_count",
    "skill_match_count",
    "skill_match_ratio",
    "seniority_match",
    "industry_match",
    "seniority_resume_encoded",
    "seniority_job_encoded"
]

numeric_data = data[numeric_columns].copy()
numeric_data = numeric_data.fillna(0)

print(numeric_data.head())


In [ ]:
scaler = StandardScaler()

numeric_scaled = scaler.fit_transform(numeric_data)

numeric_scaled = pd.DataFrame(
    numeric_scaled,
    columns=numeric_columns
)

print("Feature scaling completed.")


In [ ]:
features = pd.concat(
    [
        numeric_scaled.reset_index(drop=True),
        encoded_data.reset_index(drop=True)
    ],
    axis=1
)

target = data["target"].reset_index(drop=True)

print("Feature shape:", features.shape)
print("Target shape:", target.shape)


In [ ]:
correlation_data = features.copy()
correlation_data["target"] = target

correlation = correlation_data.corr()["target"].sort_values(ascending=False)

print(correlation)


In [ ]:
plt.figure(figsize=(10, 6))

sns.heatmap(
    correlation_data[
        [
            "years_experience",
            "skill_count",
            "required_skill_count",
            "skill_match_count",
            "skill_match_ratio",
            "seniority_match",
            "industry_match",
            "target"
        ]
    ].corr(),
    annot=True,
    cmap="coolwarm"
)

plt.title("Correlation of Engineered Features")
plt.show()


In [ ]:
k = min(20, features.shape[1])

selector = SelectKBest(
    score_func=f_classif,
    k=k
)

selected_features = selector.fit_transform(
    features,
    target
)

selected_columns = features.columns[selector.get_support()]

print("Selected features:")
print(selected_columns.tolist())


In [ ]:
final_dataset = pd.DataFrame(
    selected_features,
    columns=selected_columns
)

final_dataset["target"] = target.values

print("Final dataset shape:", final_dataset.shape)
display(final_dataset.head())


In [ ]:
print("Final Dataset Shape:", final_dataset.shape)

print("\nMissing Values:")
print(final_dataset.isnull().sum().sum())

print("\nDuplicate Rows:")
print(final_dataset.duplicated().sum())

print("\nTarget Distribution:")
print(final_dataset["target"].value_counts())


In [ ]:
final_dataset.to_csv(
    "ATS_Resume_Matcher_Final_Dataset.csv",
    index=False
)

print("Final dataset saved successfully.")
